## **Reading JSON File as DataFrame**

In [1]:
df = spark.read.option("multiline", "true").json("Files/bing-latest-news.json")
# df now is a Spark DataFrame containing JSON data from "Files/bing-latest-news.json".
display(df)

StatementMeta(, c92f6c28-b8c4-4862-8d53-a557b2af055b, 3, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 722f97f2-0d5a-497a-b257-d486973680ac)

#### Selecting Value (Necessary) Column

In [2]:
df = df.select("value")
display(df)

StatementMeta(, c92f6c28-b8c4-4862-8d53-a557b2af055b, 4, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 434e905e-50ec-4fc7-a709-1fb4c0b1e7cf)

#### Applying Explode function on JSON column

In [3]:
from pyspark.sql.functions import explode
df_exploded = df.select(explode(df["value"]).alias("json_object"))
display(df_exploded)

StatementMeta(, c92f6c28-b8c4-4862-8d53-a557b2af055b, 5, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 739f3e87-aad5-4d8a-b838-8d6b24a52660)

#### Compile the Exploded elements in the form of the list to traverse through each


In [4]:
json_list = df_exploded.toJSON().collect()


StatementMeta(, c92f6c28-b8c4-4862-8d53-a557b2af055b, 6, Finished, Available, Finished)

In [5]:
print(json_list[0])

StatementMeta(, c92f6c28-b8c4-4862-8d53-a557b2af055b, 7, Finished, Available, Finished)

{"json_object":{"about":[{"name":"Triumph TR5","readLink":"https://api.bing.microsoft.com/api/v7/entities/ba7c9eb3-0760-8c01-fbfd-a01f783d390b"},{"name":"North American XF-108 Rapier","readLink":"https://api.bing.microsoft.com/api/v7/entities/52130234-292e-907e-6d10-94d261a80425"},{"name":"Cold War","readLink":"https://api.bing.microsoft.com/api/v7/entities/74f04e44-5b11-b73d-c19a-2090f768e113"},{"name":"Taylor Swift","readLink":"https://api.bing.microsoft.com/api/v7/entities/98004a38-a3ea-b902-b6d3-687b30b54353"},{"name":"Travis Kelce","readLink":"https://api.bing.microsoft.com/api/v7/entities/4e917ff4-3613-42f8-bf79-9c12138e3ede"},{"name":"Super Bowl","readLink":"https://api.bing.microsoft.com/api/v7/entities/ce1fece8-34c4-6249-a1aa-2e779294760e"},{"name":"Bangkok","readLink":"https://api.bing.microsoft.com/api/v7/entities/651e796f-a780-c032-4e9c-23e5f5fe5854"},{"name":"Thailand","readLink":"https://api.bing.microsoft.com/api/v7/entities/588bd4b9-e440-b7eb-2cab-2a54c0458548"}],"dateP

In [6]:
import json

news_json = json.loads(json_list[24])

StatementMeta(, c92f6c28-b8c4-4862-8d53-a557b2af055b, 8, Finished, Available, Finished)

#### Testing the JSON String List

In [7]:
print(news_json)

StatementMeta(, c92f6c28-b8c4-4862-8d53-a557b2af055b, 9, Finished, Available, Finished)

{'json_object': {'category': 'Sports', 'datePublished': '2025-02-09T21:10:31.0000000Z', 'description': 'Reece Walsh has declared he has the ability to be the best player in the NRL, with the Brisbane Broncos youngster hell-bent on bouncing back in 2025. And it was revealed over the weekend that a promise made to Kevin Walters amid his departure as coach might not have eventuated.', 'name': "Reece Walsh makes staggering statement as truth about Kevin Walters' new Broncos role revealed", 'provider': [{'_type': 'Organization', 'image': {'thumbnail': {'contentUrl': 'https://www.bing.com/th?id=ODF.a1Ggn532fLWTsNQRo6BMkw&pid=news'}}, 'name': 'Yahoo Australia (English) on MSN.com'}], 'url': 'https://www.msn.com/en-au/sport/rugby_league/reece-walsh-makes-staggering-statement-as-truth-about-kevin-walters-new-broncos-role-revealed/ar-AA1yHO3A'}}


In [8]:
print(news_json['json_object']['description'])

StatementMeta(, c92f6c28-b8c4-4862-8d53-a557b2af055b, 10, Finished, Available, Finished)

Reece Walsh has declared he has the ability to be the best player in the NRL, with the Brisbane Broncos youngster hell-bent on bouncing back in 2025. And it was revealed over the weekend that a promise made to Kevin Walters amid his departure as coach might not have eventuated.


In [9]:
print(news_json['json_object']['name'])
print(news_json['json_object']['description'])
print(news_json['json_object']['category'])
print(news_json['json_object']['provider'][0]['image']['thumbnail']['contentUrl'])
print(news_json['json_object']['url'])
print(news_json['json_object']['provider'][0]['name'])
print(news_json['json_object']['datePublished'])

StatementMeta(, c92f6c28-b8c4-4862-8d53-a557b2af055b, 11, Finished, Available, Finished)

Reece Walsh makes staggering statement as truth about Kevin Walters' new Broncos role revealed
Reece Walsh has declared he has the ability to be the best player in the NRL, with the Brisbane Broncos youngster hell-bent on bouncing back in 2025. And it was revealed over the weekend that a promise made to Kevin Walters amid his departure as coach might not have eventuated.
Sports
https://www.bing.com/th?id=ODF.a1Ggn532fLWTsNQRo6BMkw&pid=news
https://www.msn.com/en-au/sport/rugby_league/reece-walsh-makes-staggering-statement-as-truth-about-kevin-walters-new-broncos-role-revealed/ar-AA1yHO3A
Yahoo Australia (English) on MSN.com
2025-02-09T21:10:31.0000000Z


#### Processing the JSON components into Lists

In [10]:
title = []
description = []
category = []
url = []
image = []
provider = []
datePublished = []

for json_str  in json_list:
    try:
        article = json.loads(json_str)

        if article['json_object'].get('category') and article['json_object']['provider'][0].get('image',{}).get('thumbnail',{}).get('contentUrl'):

            title.append(article['json_object']['name'])
            description.append(article['json_object']['description'])
            category.append(article['json_object']['category'])
            url.append(article['json_object']['url'])
            image.append(article['json_object']['provider'][0]['image']['thumbnail']['contentUrl'])
            provider.append(article['json_object']['provider'][0]['name'])
            datePublished.append(article['json_object']['datePublished'])

    except Exception as e:
        print(f"Error processing JSON object: {e}")

StatementMeta(, c92f6c28-b8c4-4862-8d53-a557b2af055b, 12, Finished, Available, Finished)

In [11]:
title

StatementMeta(, c92f6c28-b8c4-4862-8d53-a557b2af055b, 13, Finished, Available, Finished)

["Latest Car and Bike News Live Updates Today February 10, 2025: Ola Roadster X electric motorcycle aims to spearhead India's EV offensive",
 'Daily Briefing: Biren Singh’s day of reckoning; the search for Delhi CM',
 'Musk charges on with new targets in sight and Trump’s blessing',
 'WNBA free agency and trade tracker 2025: Deals, news, moves',
 'Sudan army plans new government as it advances in capital',
 'LISTEN: AFL Daily is BACK, Dogs latest, fallout at Saints',
 'New writing needs protection from AI, says Makar',
 'The latest on Donald Trump’s presidency',
 "Trump at Super Bowl LIX in New Orleans: List of family members who did (and didn't) attend",
 "Reece Walsh makes staggering statement as truth about Kevin Walters' new Broncos role revealed",
 'New arts hub appoints its first chief executive',
 'Liverpool news: Arne Slot takes aim at players for FA Cup shock as Reds star bemoans ref',
 'Boston Celtics vs. Miami Heat Injury Report: News, Statuses, Inactives for Monday, Februar

#### Converting the Lists into a DF using Spark

In [12]:
from pyspark.sql.types import StructType, StructField, StringType

data = list(zip(title,description,category,url,image,provider,datePublished))

schema = StructType([
    StructField("title", StringType(), True),
    StructField("descrption", StringType(), True),
    StructField("category", StringType(), True),
    StructField("url", StringType(), True),
    StructField("image", StringType(), True),
    StructField("provider", StringType(), True),
    StructField("datePublished", StringType(), True)
])

df_cleaned = spark.createDataFrame(data, schema = schema)

StatementMeta(, c92f6c28-b8c4-4862-8d53-a557b2af055b, 14, Finished, Available, Finished)

In [13]:
display(df_cleaned)

StatementMeta(, c92f6c28-b8c4-4862-8d53-a557b2af055b, 15, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 2ca3d8e0-05ab-4a57-814b-02283ed000d7)

#### Converting the DatePublished column into usable format

In [14]:
from pyspark.sql.functions import to_date, date_format

df_cleaned_final = df_cleaned.withColumn("datePublished", date_format(to_date("datePublished"), "dd-MMM-yyyy"))

StatementMeta(, c92f6c28-b8c4-4862-8d53-a557b2af055b, 16, Finished, Available, Finished)

In [15]:
display(df_cleaned_final)

StatementMeta(, c92f6c28-b8c4-4862-8d53-a557b2af055b, 17, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 50cf3aca-b046-4560-b071-4735b516667e)

#### Saving Final table into Database

In [20]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bing_lake_db")

StatementMeta(, c92f6c28-b8c4-4862-8d53-a557b2af055b, 22, Finished, Available, Finished)

DataFrame[]

In [21]:
df_cleaned_final.write.format("delta").saveAsTable("bing_lake_db.tbl_latest_news")

StatementMeta(, c92f6c28-b8c4-4862-8d53-a557b2af055b, 23, Finished, Available, Finished)

In [ ]:
%%sql

select count(*) from bing_lake_db.tbl_latest_news

StatementMeta(, c92f6c28-b8c4-4862-8d53-a557b2af055b, -1, Cancelled, , Cancelled)

#### For Incremental Load using Type1 Merge for updation

In [ ]:
from pyspark.sql.utils import AnalysisException

try:
    table_name = 'bing_lake_db.tbl_latest_news'
    df_cleaned_final.write.format("delta").saveAsTable(table_name)

except AnalysisException:
    print("Table Already Exists")

    df_cleaned_final.createOrReplaceTempView("vw_df_cleaned_final")

    spark.sql(f""" MERGE INTO {table_name} target_table
                    USING vw_df_cleaned_final source_view

                    ON source_view.url = target_table.url

                    WHEN MATCHED AND
                    source_view.title <> target_table.title OR
                    source_view.descrption <> target_table.descrption OR
                    source_view.category <> target_table.category OR
                    source_view.image <> target_table.image OR
                    source_view.provider <> target_table.provider OR
                    source_view.datePublished <> target_table.datePublished 
                    
                    THEN UPDATE SET *
                    WHEN NOT MATCHED THEN INSERT *
                """)

StatementMeta(, c92f6c28-b8c4-4862-8d53-a557b2af055b, -1, Cancelled, , Cancelled)